<a href="https://colab.research.google.com/github/xwx-codingweb/XIA_DSPN_S26/blob/master/ExerciseSubmissions/10_mixed-effects-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 10: Mixed effects

This homework assignment is designed to give you practice fitting and interpreting mixed effects models.

We will be using the **LexicalData.csv** and **Items.csv** files from the *Homework/lexDat* folder in the class GitHub repository again.

This data is a subset of the [English Lexicon Project database](https://elexicon.wustl.edu/). It provides the reaction times (in milliseconds) of many subjects as they are presented with letter strings and asked to decide, as quickly and as accurately as possible, whether the letter string is a word or not. The **Items.csv** provides characteristics of the words used, namely frequency (how common is this word?) and length (how many letters?). Unlike in the previous homework, there isn't any missing data in the **LexicalData.csv** file.

*Data courtesy of Balota, D.A., Yap, M.J., Cortese, M.J., Hutchison, K.A., Kessler, B., Loftis, B., Neely, J.H., Nelson, D.L., Simpson, G.B., & Treiman, R. (2007). The English Lexicon Project. Behavior Research Methods, 39, 445-459.*

---
## 1. Loading and formatting the data (1 point)

Load in data from the **LexicalData.csv** and **Items.csv** files. As in the previous homeworks, remove the commas from the reaction times and convert them from strings to numbers. Use `left_join` to add word characteristics `Length` and `Log_Freq_Hal` from **Items** to **LexicalData**.

*Note: the `Freq_HAL` variable in **Items.csv** has a similar formatting issue, using string values with commas. We're not going to worry about fixing this since we're only using `Log_Freq_HAL`, which is the natural log transformation of `Freq_HAL`, in this homework.*

In [10]:
system("git clone https://github.com/robjavvar/DSPN_CourseNotebook.git")
list.files()

library(tidyverse)

[1] "DSPN_CourseNotebook" "sample_data"

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rbibutils’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘numDeriv’


Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack



Attaching package: ‘lmerTest’


The following object is masked from ‘package:lme4’:

    lmer


The following object is masked from ‘package:stats’:

    step




In [7]:
getwd()


files <- list.files()

files

lex_path <- list.files("DSPN_CourseNotebook", pattern = "LexicalData\\.csv$",
                       recursive = TRUE, full.names = TRUE)
items_path <- list.files("DSPN_CourseNotebook", pattern = "Items\\.csv$",
                         recursive = TRUE, full.names = TRUE)

lex_path
items_path


lex <- read.csv(lex_path[1], stringsAsFactors = FALSE)
items <- read.csv(items_path[1], stringsAsFactors = FALSE)

head(lex)
head(items)

[1] "/content"

[1] "DSPN_CourseNotebook" "sample_data"

[1] "DSPN_CourseNotebook/book/exercises/lexDat/LexicalData.csv"

[1] "DSPN_CourseNotebook/book/exercises/lexDat/Items.csv"

,Sub_ID,Trial,Type,D_RT,D_Word,Outlier,D_Zscore
,<int>,<int>,<int>,<chr>,<chr>,<chr>,<dbl>
1,157,1,1,710,browse,false,-0.437
2,67,1,1,"1,094",refrigerant,false,0.825
3,120,1,1,587,gaining,false,-0.645
4,21,1,1,984,cheerless,false,0.025
5,236,1,1,577,pattered,false,-0.763
6,236,2,1,715,conjures,false,-0.364


,Occurrences,Word,Length,Freq_HAL,Log_Freq_HAL
,<int>,<chr>,<int>,<chr>,<dbl>
1,1,synergistic,11,284,5.649
2,1,synonymous,10,951,6.858
3,1,syntactical,11,114,4.736
4,1,synthesis,9,"6,742",8.816
5,1,synthesized,11,"2,709",7.904
6,1,synthesizer,11,"1,390",7.237


In [8]:
# WRITE YOUR CODE HERE
# Remove commas from RT and convert to numeric
lex <- lex %>%
  mutate(D_RT = as.numeric(gsub(",", "", D_RT)))

# Join Length and Log_Freq_HAL from Items to LexicalData
lexical_data <- lex %>%
  left_join(
    items %>% select(Word, Length, Log_Freq_HAL),
    by = c("D_Word" = "Word"))%>%
    drop_na()


head(lexical_data)

,Sub_ID,Trial,Type,D_RT,D_Word,Outlier,D_Zscore,Length,Log_Freq_HAL
,<int>,<int>,<int>,<dbl>,<chr>,<chr>,<dbl>,<int>,<dbl>
1,157,1,1,710,browse,false,-0.437,6,8.856
2,67,1,1,1094,refrigerant,false,0.825,11,4.644
3,120,1,1,587,gaining,false,-0.645,7,8.304
4,21,1,1,984,cheerless,false,0.025,9,2.639
5,236,1,1,577,pattered,false,-0.763,8,1.386
6,236,2,1,715,conjures,false,-0.364,8,5.268


---
## 2. Model fitting (4 points)

First, fit a linear model with `Log_Freq_HAL` and `Length` as predictors, and `D_RT` as the output. Include an interaction term. Use `summary()` to look at the model output.

In [12]:
# WRITE YOUR CODE HERE
lm_model <- lm(D_RT ~ Length * Log_Freq_HAL, data = lexical_data)

summary(lm_model)


Call:
lm(formula = D_RT ~ Length * Log_Freq_HAL, data = lexical_data)

Residuals:
     Min       1Q   Median       3Q      Max 
-1118.01  -205.23   -86.95    90.77  3147.07 

Coefficients:
                    Estimate Std. Error t value Pr(>|t|)    
(Intercept)         610.1903    14.6678  41.601  < 2e-16 ***
Length               47.7531     1.6368  29.175  < 2e-16 ***
Log_Freq_HAL         -6.0239     1.9678  -3.061  0.00221 ** 
Length:Log_Freq_HAL  -2.9421     0.2348 -12.528  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 359.1 on 62606 degrees of freedom
Multiple R-squared:  0.09473,	Adjusted R-squared:  0.09469 
F-statistic:  2184 on 3 and 62606 DF,  p-value: < 2.2e-16


Now, install `lme4` using `install.packages()` and then load the library.

In [ ]:
# WRITE YOUR CODE HERE
install.packages("lme4")
install.packages("lmerTest")

library(lme4)
library(lmerTest)

Now fit a mixed effects model that includes the same predictors as the linear model above, as well as random intercepts for `Sub_ID` (i.e., cases where subject ID shifts the RT mean). Use `summary()` to look at the model output.

In [19]:
# WRITE YOUR CODE HERE
mixed_model <- lmer(
  D_RT ~ Length * Log_Freq_HAL + (1 | Sub_ID),
  data = lexical_data
)

summary(mixed_model)

Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: D_RT ~ Length * Log_Freq_HAL + (1 | Sub_ID)
   Data: lexical_data

REML criterion at convergence: 888235.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.5058 -0.5472 -0.1568  0.3103 10.7381 

Random effects:
 Groups   Name        Variance Std.Dev.
 Sub_ID   (Intercept) 46333    215.3   
 Residual             82978    288.1   
Number of obs: 62610, groups:  Sub_ID, 299

Fixed effects:
                      Estimate Std. Error         df t value Pr(>|t|)    
(Intercept)           616.8445    17.1522  1051.7517  35.963  < 2e-16 ***
Length                 47.7477     1.3162 62313.3658  36.277  < 2e-16 ***
Log_Freq_HAL           -7.4374     1.5830 62314.1248  -4.698 2.63e-06 ***
Length:Log_Freq_HAL    -2.8778     0.1888 62313.3661 -15.239  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
            (Intr) Length L_F_HA

---
## 3. Model assessment (4 points)

Compare the three t-values for the fixed effects and the mixed effects models. How do they differ, and why?

> The t-values for the mixed-effects model are smaller than those from the linear model.

> This happens because the mixed-effects model accounts for variability across subjects using random intercepts.

> By modeling this additional variance, the estimates of the fixed effects become more conservative and realistic, which often reduces the magnitude of the t-values.

Use the Aikeke Information Criterion (AIC) to compare these two models. Which one is better?

In [20]:
# WRITE YOUR CODE HERE

AIC(lm_model, mixed_model)

,df,AIC
,<dbl>,<dbl>
lm_model,5,914436.4
mixed_model,6,888247.6


> The mixed-effects model is better because it has a lower AIC value, indicating a better fit while accounting for model complexity (subject-level variation in RT).


---
##  4. Reflection (1 point)

What other random effects could be controlled for in this data set?

> D_Word. Different words may have different processing times.
>

**DUE:** 5pm EST, March 5, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*